In [1]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)

# LLM Agents

This notebook introduces agents and a simple Function Calling workflow. An agent
perceives an environment and acts toward a goal. We will connect a model to a
weather function, compare responses with and without tools, then add a second tool.

English | [Simplified Chinese](../../01_function-calling/01_fc-practice.ipynb)

All model examples read `vLLM_QWEN_BASE_URL`, `vLLM_QWEN_API_KEY` and
`vLLM_QWEN_MODEL_ID` from the repository-root `.env`. Configure weather credentials
before making live weather calls.
The Chinese edition linked above is the master text. This notebook was executed with the
`env_agent_uv` kernel, so the saved outputs come from a real run; model wording changes
between runs and weather values are live.

<img src="../../../../assets/agent1.png" alt="Agent perception and action loop" width="90%"/>


> This notebook was executed with the `env_agent_uv` kernel (Python 3.12) on 2026-10-18 and the outputs are saved. Every code cell is followed by an **Expected output:** block, so the real results stay readable after Clear All Outputs; model wording and live weather values change between runs.

## What is an agent?

An agent uses sensors to observe an environment and actuators to take action.
Sensors may be cameras, events or API responses; actuators may be physical devices,
code or service calls. The environment is the world in which the task occurs.
Autonomy describes the decisions delegated to the system within its constraints.
Observation, decision, action and feedback form the agent loop.

## Traditional agents

Simple reflex agents use condition/action rules. A thermostat switches cooling on
when the observed temperature exceeds its threshold. Such rules are efficient but
cannot account for missing context unless the system also maintains state. Other
designs add internal models, explicit goals, utility functions or learning.

<img src="../../../../assets/agent2.png" alt="Traditional agent architecture" width="90%"/>

## LLM-driven agents

An LLM can interpret natural-language goals, propose a plan and select tools. The
application supplies execution, state and controls. Flexibility does not guarantee
correct decisions, so validate proposed actions and the results they return.

For a trip to Xiamen, the agent can clarify preferences, check weather, draft an
itinerary and revise it when the budget changes. A booking remains a separate action.
The cycle is: identify a missing fact, call a tool, inspect the result and update the plan.

<img src="../../../../assets/agent3.png" alt="Comparison of traditional and LLM agent designs" width="90%"/>

Original course figures may contain Chinese labels; the surrounding text explains them.

# Function Calling

## Introduction

Models cannot obtain live weather from training data alone. Function Calling lets
an application provide named operations and argument schemas. The model requests a
call; application code executes it and returns the result for a grounded answer.

OpenAI introduced its API Function Calling feature in June 2023. The mechanism is
now used by multiple model providers. MCP can standardize the external capability
interface while a host still uses model tool calling to choose operations.

The basic exchange is shown below.

<img src="../../../../assets/FC01.png" alt="Function Calling exchange" style="width:50%;" />

For live weather, implement an HTTP-query function and describe its input to the model. The model can then request the city query when needed.

<img src="../../../../assets/FC04.png" alt="Weather function integration" style="width:50%;" />

## How models learn to call tools

Tool-call training examples teach models to select function names and produce structured arguments. Support depends on the exact model, deployment and API. Check your provider's supported model version rather than assuming that all reasoning or chat models support the same interface.

<img src="../../../../assets/FC03.png" alt="Tool-call training example" style="width:80%;" />

## Configure a model

All model calls use `my_qwen3.8_flash_next`, configured in the repository-root `.env`.

### Load the shared model configuration

In [2]:
import os
from dotenv import load_dotenv, find_dotenv




_ = load_dotenv(find_dotenv(usecwd=True))


base_url = os.environ["vLLM_QWEN_BASE_URL"]
api_key = os.environ["vLLM_QWEN_API_KEY"]

from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url)

response = client.chat.completions.create(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    messages=[
        {"role": "system", "content": "You are a weather assistant."},
        {"role": "user", "content": "Who are you?"}
    ],
    stream=False
)

print(response.choices[0].message.content)



I’m your weather assistant, here to help with weather-related questions—like forecasts, temperature, wind, rain, humidity, and weather alerts.


**Expected output:**

```text
I’m your weather assistant, here to help with weather-related questions—like forecasts, temperature, wind, rain, humidity, and weather alerts.
```


## Design the function

### Test the OpenWeather API

An API is an application programming interface. Here the OpenAI-compatible Chat Completions API exchanges model messages; a separate OpenWeather HTTP API retrieves weather data.

Create an API key through [OpenWeather](https://openweathermap.org/). Set `OPENWEATHER_API_KEY` and `OPENWEATHER_BASE_URL` in your local environment. For this notebook the URL is the complete current-weather endpoint: `https://api.openweathermap.org/data/2.5/weather`. Do not print or commit your key.

In [3]:
import os
from dotenv import load_dotenv, find_dotenv






_ = load_dotenv(find_dotenv(usecwd=True))

In [4]:
find_dotenv(usecwd=True)

'/Users/will/workspace/llm-agent-lab/.env'

**Expected output:**

```text
'/Users/will/workspace/llm-agent-lab/.env'
```


In [5]:
open_weather_key=os.environ["OPENWEATHER_API_KEY"]
open_weather_url=os.environ["OPENWEATHER_BASE_URL"]

Call the current-weather endpoint.

In [6]:


import requests


open_weather_url=os.environ["OPENWEATHER_BASE_URL"]
open_weather_key=os.environ["OPENWEATHER_API_KEY"]
# url = "https://api.openweathermap.org/data/2.5/weather"


params = {
    "q": "Beijing",               
    "appid": open_weather_key,    
    "units": "metric",            
    "lang":"en"                
}


response = requests.get(open_weather_url, params=params)


data = response.json()

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Request failed; status code:{response.status_code}")

{'coord': {'lon': 116.3972, 'lat': 39.9075}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 14.03, 'feels_like': 12.12, 'temp_min': 14.03, 'temp_max': 14.03, 'pressure': 1022, 'humidity': 24, 'sea_level': 1022, 'grnd_level': 1017}, 'visibility': 10000, 'wind': {'speed': 1.76, 'deg': 165, 'gust': 3.04}, 'clouds': {'all': 1}, 'dt': 1792334697, 'sys': {'country': 'CN', 'sunrise': 1792276028, 'sunset': 1792315911}, 'timezone': 28800, 'id': 1816670, 'name': 'Beijing', 'cod': 200}


**Expected output:**

```text
{'coord': {'lon': 116.3972, 'lat': 39.9075}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 14.03, 'feels_like': 12.12, 'temp_min': 14.03, 'temp_max': 14.03, 'pressure': 1022, 'humidity': 24, 'sea_level': 1022, 'grnd_level': 1017}, 'visibility': 10000, 'wind': {'speed': 1.76, 'deg': 165, 'gust': 3.04}, 'clouds': {'all': 1}, 'dt': 1792334697, 'sys': {'country': 'CN', 'sunrise': 1792276028, 'sunset': 1792315911}, 'timezone': 28800, 'id': 1816670, 'name': 'Beijing', 'cod': 200}
```


See [OpenWeather response fields](https://openweathermap.org/current#current_JSON) for units and field meanings.

### Implement `get_weather`

Wrap the HTTP request in a Python function. This exercise accepts a city string and returns serialized JSON, which can be placed in a tool-result message. Tool arguments in general can have other JSON-compatible types.

In [7]:
import json

def get_weather(loc):
    """Return current weather JSON for a city recognized by OpenWeather."""
    
    open_weather_key = os.environ["OPENWEATHER_API_KEY"]
    open_weather_url = os.environ["OPENWEATHER_BASE_URL"]

    
    params = {
        "q": loc,                     
        "appid": open_weather_key,    
        "units": "metric",            
        "lang": "en"               
    }

    
    response = requests.get(open_weather_url, params=params)
    
    
    data = response.json()
    return json.dumps(data, ensure_ascii=False)  

Test the function directly before connecting it to the model.

In [8]:
get_weather('Beijing')

'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 800, "main": "Clear", "description": "clear sky", "icon": "01n"}], "base": "stations", "main": {"temp": 14.03, "feels_like": 12.12, "temp_min": 14.03, "temp_max": 14.03, "pressure": 1022, "humidity": 24, "sea_level": 1022, "grnd_level": 1017}, "visibility": 10000, "wind": {"speed": 1.76, "deg": 165, "gust": 3.04}, "clouds": {"all": 1}, "dt": 1792334697, "sys": {"country": "CN", "sunrise": 1792276028, "sunset": 1792315911}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'

**Expected output:**

```text
'{"coord": {"lon": 116.3972, "lat": 39.9075}, "weather": [{"id": 800, "main": "Clear", "description": "clear sky", "icon": "01n"}], "base": "stations", "main": {"temp": 14.03, "feels_like": 12.12, "temp_min": 14.03, "temp_max": 14.03, "pressure": 1022, "humidity": 24, "sea_level": 1022, "grnd_level": 1017}, "visibility": 10000, "wind": {"speed": 1.76, "deg": 165, "gust": 3.04}, "clouds": {"all": 1}, "dt": 1792334697, "sys": {"country": "CN", "sunrise": 1792276028, "sunset": 1792315911}, "timezone": 28800, "id": 1816670, "name": "Beijing", "cod": 200}'
```


### Describe the function in the tools list

In [9]:
tools = [
    {
        "type": "function",     
        "function": {
            "name": "get_weather",  
            "description": "Get current weather for a city. Return the OpenWeather JSON response, including available temperature, conditions, humidity, wind and observation time.",  
            "parameters": {
                "type": "object",    
                "properties": {
                    "location": {
                        "type": "string",  
                        "description": "City name, for example Beijing or Xian. Use a name recognized by OpenWeather.",  
                    }
                },
                "required": ["location"]  
            },
        }
    },
]

## Without tools versus Function Calling

### Without tools

First send a weather question without providing a tools list. The model has no live weather source in this exchange.

In [10]:
from openai import OpenAI

base_url = os.environ["vLLM_QWEN_BASE_URL"]
api_key = os.environ["vLLM_QWEN_API_KEY"]

client = OpenAI(api_key=api_key, base_url=base_url)

response = client.chat.completions.create(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    messages=[
    {"role": "user", "content": "Please check the weather in Xian. /no_think"}
    ]
)

In [11]:
response

ChatCompletion(id='chatcmpl-9ae59380793f5281', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nI can’t access real-time weather information right now. For the current weather in Xi’an, please check a local weather service or search “Xi’an weather” on your preferred search engine.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to respond to user: "Please check the weather in Xian. /no_think". They explicitly request /no_think maybe indicating no thinking? But must obey? We need final only? Need answer weather. But no web browsing tool. We can state cannot access real-time weather, suggest checking local forecast? Need maybe concise. Also mention Xi\'an spelling? Ensure final in English.\n'), stop_reason=None, token_ids=None, routed_experts=None)], created=1792334971, model='my_qwen3.8_flash_next', object='chat.completion', moderation=None, service_tier=None, 

**Expected output:**

```text
ChatCompletion(id='chatcmpl-9ae59380793f5281', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nI can’t access real-time weather information right now. For the current weather in Xi’an, please check a local weather service or search “Xi’an weather” on your preferred search engine.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to respond to user: "Please check the weather in Xian. /no_think". They explicitly request /no_think maybe indicating no thinking? But must obey? We need final only? Need answer weather. But no web browsing tool. We can state cannot access real-time weather, suggest checking local forecast? Need maybe concise. Also mention Xi\'an spelling? Ensure final in English.\n'), stop_reason=None, token_ids=None, routed_experts=None)], created=1792334971, model='my_qwen3.8_flash_next', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='vllm-0.31.1rc1.dev173+g8cbd5d030-tp8-ep-960078d0', usage=CompletionUsage(completion_tokens=123, prompt_tokens=64, total_tokens=187, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=82, rejected_prediction_tokens=None), prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, prompt_text=None, kv_transfer_params=None, ec_transfer_params=None, metrics=None)
```


A formatted response makes the absence of tool calls easier to see. This shortened example is illustrative:

```json
{"choices":[{"message":{"role":"assistant","content":"I need a live weather source to answer this request.","tool_calls":null}}]}
```

The response does not contain retrieved weather data.

Inspect the `tool_calls` field.

In [12]:
print(response.choices[0].message.tool_calls)

None


**Expected output:**

```text
None
```


If `tool_calls` is `None`, the model did not request a function.

### With Function Calling

Reuse the weather function:

In [13]:
import json

def get_weather(loc):
    """Return current weather JSON for a city recognized by OpenWeather."""
    
    open_weather_key = os.environ["OPENWEATHER_API_KEY"]
    open_weather_url = os.environ["OPENWEATHER_BASE_URL"]

    
    params = {
        "q": loc,                     
        "appid": open_weather_key,    
        "units": "metric",            
        "lang": "en"               
    }

    
    response = requests.get(open_weather_url, params=params)
    
    
    data = response.json()
    return json.dumps(data, ensure_ascii=False)  

Provide its full tool definition:

In [14]:
tools = [
    {
        "type": "function",     
        "function": {
            "name": "get_weather",  
            "description": "Get current weather for a city. Return the OpenWeather JSON response, including available temperature, conditions, humidity, wind and observation time.",  
            "parameters": {
                "type": "object",    
                "properties": {
                    "location": {
                        "type": "string",  
                        "description": "City name, for example Beijing or Xian. Use a name recognized by OpenWeather.",  
                    }
                },
                "required": ["location"]  
            },
        }
    },
]

Now include the tool definition in Chat Completions. See the [OpenAI guide](https://developers.openai.com/api/docs/guides/function-calling) and your provider's tool-calling documentation. This notebook uses Chat Completions messages; Responses API call/result items have a different format.

The application handles the following steps:

1. Receive the weather question.
2. Let the model select the weather tool and extract the city.
3. Parse and validate the arguments.
4. Call OpenWeather through the Python function.
5. Append the assistant call and its matching tool result.
6. Request a user-facing answer based on the returned data.

In [15]:
import json
import requests
from openai import OpenAI
from datetime import datetime
import os
from dotenv import load_dotenv, find_dotenv



_ = load_dotenv(find_dotenv(usecwd=True))

# Your OpenWeather API key
# open_weather_key=os.environ["API_KEY_OPENWEATHER"]
# open_weather_url=os.environ["BASE_URL_OPENWEATHER"]


def get_weather(loc):
    """Return current weather JSON for a city recognized by OpenWeather."""
    
    open_weather_url = os.environ["OPENWEATHER_BASE_URL"]
    open_weather_key = os.environ["OPENWEATHER_API_KEY"]

    
    params = {
        "q": loc,               
        "appid": open_weather_key,    
        "units": "metric",            
        "lang": "en"               
    }

    
    response = requests.get(open_weather_url, params=params)
    
    
    data = response.json()
    return json.dumps(data, ensure_ascii=False)  


tools = [
    {
        "type": "function",     
        "function": {
            "name": "get_weather",  
            "description": "Get current weather for a city. Return the OpenWeather JSON response, including available temperature, conditions, humidity, wind and observation time.",  
            "parameters": {
                "type": "object",    
                "properties": {
                    "location": {
                        "type": "string",  
                        "description": "City name, for example Beijing or Xian. Use a name recognized by OpenWeather.",  
                    }
                },
                "required": ["location"]  
            },
        }
    },
]



def send_messages(messages):
    """Send message history and return the first assistant message."""

    base_url = os.environ["vLLM_QWEN_BASE_URL"]
    api_key = os.environ["vLLM_QWEN_API_KEY"]
    client = OpenAI(
        api_key=api_key,
        base_url=base_url,
    )

    response = client.chat.completions.create(
        model=os.environ["vLLM_QWEN_MODEL_ID"],  
        messages=messages,      
        tools=tools             
    )
    return response.choices[0].message  




if __name__ == "__main__":
    messages = [{"role": "user", "content": "What is the weather in Xian today?"}]  
    print(f"User>\t {messages[0]['content']}")  
    
    # First API call to get function request
    message = send_messages(messages)  
    
    if message.tool_calls:  
        
        tool = message.tool_calls[0]  
        tool_name = tool.function.name  
        tool_arguments = json.loads(tool.function.arguments)  
        location = tool_arguments["location"]  
    
        print(f"Model>\t 🚨 Calling Tool: **{tool_name}** with arguments: {tool.function.arguments}")
        # -----------------------------
    
        messages.append(message)  
    
        # Extract location and call weather function
        weather_result = get_weather(location)  
    
        # Add function result to messages
        messages.append({
            "role": "tool",              
            "tool_call_id": tool.id,     
            "content": weather_result    
        })
    
        # Second API call to get final response
        message = send_messages(messages)  
        print(f"Model>\t {message.content}")  
    else:
        print(f"Model>\t {message.content}")  

User>	 What is the weather in Xian today?


Model>	 🚨 Calling Tool: **get_weather** with arguments: {"location": "Xian"}


Model>	 

Xi'an currently has **light rain** conditions.

- **Temperature:** 13.7°C (feels like 12.6°C)
- **Conditions:** Light rain
- **Humidity:** 58%
- **Wind:** 2.3 m/s (gentle breeze)
- **Pressure:** 1021 hPa
- **Visibility:** 10 km
- **Cloud cover:** 96%

With the rain and overcast skies, an umbrella or raincoat would be a good idea if you're heading out today.


**Expected output:**

```text
User>	 What is the weather in Xian today?

Model>	 🚨 Calling Tool: **get_weather** with arguments: {"location": "Xian"}

Model>	 

Xi'an currently has **light rain** conditions.

- **Temperature:** 13.7°C (feels like 12.6°C)
- **Conditions:** Light rain
- **Humidity:** 58%
- **Wind:** 2.3 m/s (gentle breeze)
- **Pressure:** 1021 hPa
- **Visibility:** 10 km
- **Cloud cover:** 96%

With the rain and overcast skies, an umbrella or raincoat would be a good idea if you're heading out today.
```


Illustrative output; temperatures are example data, not a live observation:

```text
User> What is the weather in Xian today?
Model> Calling Tool: get_weather with arguments: {"location": "Xian"}
Model> Xian is partly cloudy, with a temperature of 10.45 degrees Celsius.
```

Add a second tool, `get_person_info`, to observe tool selection. Its person records are static demo data.

In [16]:
import json
import requests
from openai import OpenAI
from datetime import datetime
import os
from dotenv import load_dotenv, find_dotenv



_ = load_dotenv(find_dotenv(usecwd=True))



def get_weather(loc):
    """Return current weather JSON for a city recognized by OpenWeather."""
    
    open_weather_url = os.environ["OPENWEATHER_BASE_URL"]
    open_weather_key = os.environ["OPENWEATHER_API_KEY"]

    
    params = {
        "q": loc,               
        "appid": open_weather_key,    
        "units": "metric",            
        "lang": "en"               
    }

    
    response = requests.get(open_weather_url, params=params)
    
    
    data = response.json()
    return json.dumps(data, ensure_ascii=False)  

def get_person_info(person_name):
    """Return static demonstration information for a person name."""
    
    
    example_person_info = {
        "name": person_name,
        "occupation": "Software Engineer",
        "location": "Beijing, China",
        "status": "Active"
    }
    return json.dumps(example_person_info, ensure_ascii=False)




tools = [
    {
        "type": "function",     
        "function": {
            "name": "get_weather",  
            "description": "Get current weather for a city. Return the OpenWeather JSON response, including available temperature, conditions, humidity, wind and observation time.",  
            "parameters": {
                "type": "object",    
                "properties": {
                    "location": {
                        "type": "string",  
                        "description": "City name, for example Beijing or Xian. Use a name recognized by OpenWeather.",  
                    }
                },
                "required": ["location"]  
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_person_info",  
            "description": "Get information about a person by name, such as their occupation or basic details. The user should supply a person's name first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "person_name": {
                        "type": "string",
                        "description": "The full name of the person, e.g. Zhang San, Li Si",
                    }
                },
                "required": ["person_name"]
            },
        }
    },
]


def send_messages(messages):
    """Send message history and return the first assistant message."""

    base_url = os.environ["vLLM_QWEN_BASE_URL"]
    api_key = os.environ["vLLM_QWEN_API_KEY"]
    client = OpenAI(
        api_key=api_key,
        base_url=base_url,
    )

    response = client.chat.completions.create(
        model=os.environ["vLLM_QWEN_MODEL_ID"],  
        messages=messages,      
        tools=tools             
    )
    return response.choices[0].message  


if __name__ == "__main__":
    messages = [{"role": "user", "content": "What is the weather in Beijing today?"}]  
    
    print(f"User>\t {messages[0]['content']}")  
    
    # First API call to get function request
    message = send_messages(messages)  
    
    if message.tool_calls:
        messages.append(message)
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_arguments = json.loads(tool_call.function.arguments)
    
            print(f"Model>\t 🚨 Calling Tool: **{tool_name}** with arguments: {tool_call.function.arguments}")
    
    
            
            if tool_name == "get_weather":
                function_result = get_weather(tool_arguments["location"])
            elif tool_name == "get_person_info":
                function_result = get_person_info(tool_arguments["person_name"])
            else:
                
                function_result = json.dumps({"error": f"Unknown function: {tool_name}"})
    
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": function_result
            })
        
        # Second API call to get final response
        message = send_messages(messages)  
        print(f"Model>\t {message.content}")  
    else:
        print(f"Model>\t {message.content}")  

User>	 What is the weather in Beijing today?


Model>	 🚨 Calling Tool: **get_weather** with arguments: {"location": "Beijing"}


Model>	 

**Beijing — Clear skies** 🌙

- **Temperature:** 14.0°C (feels like 12.1°C)
- **Conditions:** Clear sky, essentially cloud-free (1%)
- **Humidity:** 24% — quite dry
- **Wind:** Light, 1.8 m/s from the SSE, gusts to ~3.0 m/s
- **Pressure:** 1022 hPa
- **Visibility:** 10 km

Overall a pleasant, dry, and calm day with clear skies — comfortable for being outdoors, though a light jacket would be sensible given the breeze chill and low humidity.


**Expected output:**

```text
User>	 What is the weather in Beijing today?

Model>	 🚨 Calling Tool: **get_weather** with arguments: {"location": "Beijing"}

Model>	 

**Beijing — Clear skies** 🌙

- **Temperature:** 14.0°C (feels like 12.1°C)
- **Conditions:** Clear sky, essentially cloud-free (1%)
- **Humidity:** 24% — quite dry
- **Wind:** Light, 1.8 m/s from the SSE, gusts to ~3.0 m/s
- **Pressure:** 1022 hPa
- **Visibility:** 10 km

Overall a pleasant, dry, and calm day with clear skies — comfortable for being outdoors, though a light jacket would be sensible given the breeze chill and low humidity.
```


## Function Calling workflow

<img src="../../../../assets/FC08.png" alt="Function Calling workflow summary" style="width:80%;" />